# Cherry-Picked Transcript Statistics — by gene name

Gene-name-driven inspection of representative transcripts. You give a list of
**interesting gene names**; the notebook resolves them to transcripts via the
GENCODE annotation, keeps only those the pipeline flagged as **low** or **high**
entropy, and shows every pipeline's features for them.

All inputs are produced by the pipeline:

| File | Provides |
|------|----------|
| `entropy/{DATASET}_entropy_groups.tsv` | `seq_ID → entropy_group` (low/middle/high, class-separated) — *the annotation* |
| `entropy/{DATASET}_uncertainty_analysis.tsv` | per-tool entropy, `H_pred`, `I_bald`, `biotype` |
| `features/supplementary_features.tsv` | all-pipeline features (TE, non-B DNA, structure, rG4, …) |
| `resources/annotation/gencode.v47.annotation.gtf` | `transcript_id ↔ gene_name` |

Groups are labelled `low_coding` / `low_lncRNA` / `high_coding` / `high_lncRNA` /
`middle`; "low/high" means anything that is not `middle`.

## 1. Configuration — *edit this cell*

In [ ]:
from pathlib import Path

# ── The one thing you edit: interesting gene names ────────────────────────────
INTERESTING_GENES = ["EGF", "MALAT1", "SWI5", "MEG3"]

# Selected transcript IDs for interesting genes
TRANSCRIPT_IDS = {
    "EGF": "ENST00000265171",
    "MALAT1": "ENST00000710946",
    "SWI5": "ENST00000652598",
    "MEG3": "ENST00000648123"
}

# Keep only transcripts in these entropy tiers (prefix match on entropy_group).
# ("low", "high") drops the "middle" group.
KEEP_TIERS = ("low", "high")

DATASET = "gencode.v47.common.cdhit.cv"
BASE    = Path("/mnt/cbib/LNClassifier/paper")

ENT_DIR      = BASE / f"revision-wt/results/{DATASET}/features/entropy"
GROUPS_TSV   = ENT_DIR / f"{DATASET}_entropy_groups.tsv"
UNCERT_TSV   = ENT_DIR / f"{DATASET}_uncertainty_analysis.tsv"
SUPP_TSV     = BASE / f"revision-wt/results/{DATASET}/features/supplementary_features.tsv"
PRED_CSV     = BASE / f"revision-wt/results/{DATASET}/features/shap_clustered/all_predictions.csv"
GTF          = BASE / "resources/annotation/gencode.v47.annotation.gtf"

print("Inputs:")
for label, p in [("entropy groups", GROUPS_TSV), ("uncertainty", UNCERT_TSV),
                 ("supplementary", SUPP_TSV), ("predictions", PRED_CSV), ("GTF", GTF)]:
    print(f"  {label:14s}: {'OK ' if p.exists() else 'MISSING '} {p}")

Inputs:
  entropy groups: OK  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/entropy/gencode.v47.common.cdhit.cv_entropy_groups.tsv
  uncertainty   : OK  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/entropy/gencode.v47.common.cdhit.cv_uncertainty_analysis.tsv
  supplementary : OK  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/supplementary_features.tsv
  predictions   : OK  /mnt/cbib/LNClassifier/paper/revision-wt/results/gencode.v47.common.cdhit.cv/features/shap_clustered/all_predictions.csv
  GTF           : OK  /mnt/cbib/LNClassifier/paper/resources/annotation/gencode.v47.annotation.gtf


## 2. Load entropy annotations and resolve gene names

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

_strip = lambda s: s.str.replace(r"\.\d+$", "", regex=True)   # ENST….8 -> ENST…

# ── entropy_group annotation + uncertainty scores ─────────────────────────────
groups = pd.read_csv(GROUPS_TSV, sep="\t")
groups["tid"] = _strip(groups["seq_ID"])

uncert = pd.read_csv(UNCERT_TSV, sep="\t")
uncert["tid"] = _strip(uncert["seq_ID"])
uncert = uncert.set_index("tid")

# ── GTF: transcript_id -> gene_name (parse transcript lines once) ─────────────
def gene_of_transcript(gtf_path: Path) -> pd.Series:
    tid_re = re.compile(r'transcript_id "([^"]+)"')
    gn_re  = re.compile(r'gene_name "([^"]+)"')
    tid, gn = [], []
    with open(gtf_path) as fh:
        for line in fh:
            if "\ttranscript\t" not in line:
                continue
            gnm = gn_re.search(line)
            if not gnm:
                continue
            tid.append(tid_re.search(line).group(1).split(".")[0])
            gn.append(gnm.group(1))
    return pd.Series(gn, index=tid, name="gene")

gene_map = gene_of_transcript(GTF)            # index=transcript_id (no version)
print(f"GTF: {gene_map.size:,} transcripts, {gene_map.nunique():,} gene names")

GTF: 385,659 transcripts, 77,114 gene names


In [ ]:
# ── Resolve genes -> transcripts present in the dataset, keep low/high tiers ──
def resolve(genes) -> pd.DataFrame:
    wanted = {g.upper() for g in genes}
    hits = gene_map[gene_map.str.upper().isin(wanted)]
    df = (hits.rename_axis("tid").reset_index()
              .merge(groups[["tid", "entropy_group"]], on="tid", how="left"))
    df["in_dataset"] = df["entropy_group"].notna()
    df["tier"] = df["entropy_group"].str.split("_").str[0]   # low/middle/high
    return df

resolved_all = resolve(INTERESTING_GENES)
selected = resolved_all[resolved_all["tier"].isin(KEEP_TIERS)].copy()
selected = selected.sort_values(["gene", "entropy_group"]).set_index("tid")

# ── Resolution report ─────────────────────────────────────────────────────────
requested = {g.upper() for g in INTERESTING_GENES}
found     = set(resolved_all["gene"].str.upper())
missing   = sorted(requested - found)
print("Resolution report")
print("-" * 60)
for g in INTERESTING_GENES:
    sub = resolved_all[resolved_all["gene"].str.upper() == g.upper()]
    n_ds = int(sub["in_dataset"].sum())
    kept = sub["tier"].isin(KEEP_TIERS).sum()
    tiers = sub.loc[sub["in_dataset"], "entropy_group"].value_counts().to_dict()
    print(f"  {g:10s}: {len(sub):3d} annotated, {n_ds:3d} in dataset, "
          f"{kept:2d} kept (low/high) | groups={tiers}")
if missing:
    print(f"\n  NOT FOUND in GTF: {missing}")
print(f"\nSelected {len(selected)} transcript(s) across "
      f"{selected['gene'].nunique()} gene(s).")

# ── Self-check: resolver + GTF parse behave on a fixed control ────────────────
_ctrl = resolve(["MALAT1"]).set_index("tid")
assert "ENST00000710946" in _ctrl.index, "MALAT1 transcript missing from GTF parse"
assert _ctrl.loc["ENST00000710946", "entropy_group"] == "low_lncRNA", "control group changed"
print("✓ self-check passed (MALAT1 → ENST00000710946 = low_lncRNA)")

Resolution report
------------------------------------------------------------
  EGF       :  13 annotated,   4 in dataset,  4 kept (low/high) | groups={'low_coding': 4}
  MALAT1    :  72 annotated,  33 in dataset,  5 kept (low/high) | groups={'middle': 28, 'low_lncRNA': 5}
  SWI5      :   7 annotated,   6 in dataset,  1 kept (low/high) | groups={'middle': 5, 'high_coding': 1}
  MEG3      :  71 annotated,  30 in dataset,  7 kept (low/high) | groups={'middle': 23, 'high_lncRNA': 7}

Selected 17 transcript(s) across 4 gene(s).
✓ self-check passed (MALAT1 → ENST00000710946 = low_lncRNA)


In [ ]:
selected[["gene", "entropy_group"]]

,gene,entropy_group
tid,,
ENST00000265171,EGF,low_coding
ENST00000509793,EGF,low_coding
ENST00000652245,EGF,low_coding
ENST00000503392,EGF,low_coding
ENST00000710944,MALAT1,low_lncRNA
ENST00000710871,MALAT1,low_lncRNA
ENST00000710946,MALAT1,low_lncRNA
ENST00000710870,MALAT1,low_lncRNA
ENST00000710865,MALAT1,low_lncRNA


## 3. Load pipeline features (filtered to selected transcripts)

`supplementary_features.tsv` is ~0.5 GB, so we read the `seq_ID` column first to locate the wanted rows, then read only those rows.

In [ ]:
sel_ids = set(selected.index)

# pass 1: row positions of wanted transcripts
seqcol = pd.read_csv(SUPP_TSV, sep="\t", usecols=["seq_ID"])["seq_ID"]
keep_pos = set(np.flatnonzero(_strip(seqcol).isin(sel_ids)))   # 0-based data rows

# pass 2: read header + only those rows
feat = pd.read_csv(SUPP_TSV, sep="\t",
                   skiprows=lambda i: i > 0 and (i - 1) not in keep_pos)
feat["tid"] = _strip(feat["seq_ID"])
feat = feat.set_index("tid")
feat_cols = [c for c in feat.columns if c not in ("seq_ID", "coding_class")]

print(f"Feature table: {feat.shape[0]} transcript(s) × {len(feat_cols)} features")
missing_feat = sel_ids - set(feat.index)
if missing_feat:
    print(f"  WARNING: no features for {sorted(missing_feat)}")

Feature table: 17 transcript(s) × 607 features


## 4. Selected transcripts — overview

In [ ]:
# Ensemble coding probability from cross-validated predictions.
# class_1 == coding (checked: EGF coding -> ~0.98, MALAT1/MEG3 lncRNA -> ~0.0).
pred = pd.read_csv(PRED_CSV, usecols=["seq_ID", "y_pred_proba_class_1"])
pred["tid"] = _strip(pred["seq_ID"])
coding_prob = pred.set_index("tid")["y_pred_proba_class_1"].rename("coding_prob")

overview = selected.join(
    uncert[["biotype", "H_pred", "I_bald"]], how="left"
).join(coding_prob, how="left")[
    ["gene", "entropy_group", "biotype", "coding_prob", "H_pred", "I_bald"]
]
overview.round(4)

,gene,entropy_group,biotype,coding_prob,H_pred,I_bald
tid,,,,,,
ENST00000265171,EGF,low_coding,coding,0.99,0.0491,0.0050
ENST00000509793,EGF,low_coding,coding,0.98,0.0691,0.0082
ENST00000652245,EGF,low_coding,coding,0.99,0.0721,0.0052
ENST00000503392,EGF,low_coding,coding,1.00,0.0634,0.0037
ENST00000710944,MALAT1,low_lncRNA,lncRNA,0.05,0.2636,0.0441
ENST00000710871,MALAT1,low_lncRNA,lncRNA,0.03,0.2667,0.0249
ENST00000710946,MALAT1,low_lncRNA,lncRNA,0.02,0.2508,0.0356
ENST00000710870,MALAT1,low_lncRNA,lncRNA,0.03,0.1863,0.0178
ENST00000710865,MALAT1,low_lncRNA,lncRNA,0.01,0.2702,0.0334


## 5. Entropy / uncertainty per tool

In [ ]:
# Column label for each selected transcript: "GENE (ENST… group)"
def col_label(tid: str) -> str:
    r = selected.loc[tid]
    return f"{r['gene']} ({tid} · {r['entropy_group']})"

def show(df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """Transpose so genes are columns; label columns; round."""
    return df.reindex(selected.index).T.rename(columns=col_label).round(decimals)

tool_cols = [c for c in uncert.columns if c.endswith("_entropy")]
ent_view = uncert.reindex(selected.index)[tool_cols + ["H_exp", "H_pred", "I_bald"]]
ent_view.columns = [c.replace("_entropy", "") for c in tool_cols] + ["H_exp", "H_pred", "I_bald"]
show(ent_view)

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
coding_score_rnasamba,0.0363,0.0675,0.0280,0.0594,0.1345,0.2503,0.0150,0.0548,0.2785,0.8261,0.9991,0.8278,0.9457,0.3918,0.8256,0.8501,0.3438
coding_potential_feelnc,0.0672,0.0208,0.0672,0.0938,0.0208,0.2514,0.1525,0.2141,0.0529,0.0376,0.3584,0.8763,0.2423,0.4092,0.3733,0.0376,0.3951
Coding_prob_l_cpat,0.0511,0.0405,0.0912,0.0598,0.5531,0.5363,0.5430,0.3246,0.5560,0.6523,0.3883,0.9563,0.9323,0.9017,0.2874,0.4713,0.7453
Noncoding_prob_ss_lncDC,0.0010,0.0092,0.0020,0.0550,0.0205,0.0370,0.0576,0.0025,0.0015,0.8058,0.3173,0.0030,0.0615,0.1117,0.0529,0.0825,0.0507
coding_prob_mrnn,0.1498,0.1077,0.1434,0.0825,0.1704,0.2122,0.1530,0.2143,0.2915,0.3733,0.7592,0.9728,0.6981,0.9957,0.8329,0.7398,0.1460
P(pcRNA)_lncrnabert,0.0001,0.0000,0.0000,0.0001,0.0001,0.0001,0.0039,0.0001,0.0028,0.0001,0.0076,0.0219,0.0008,0.3276,0.0002,0.0001,0.0006
prob_coding_plncpro,0.0114,0.2141,0.0808,0.1242,0.5436,0.3879,0.3986,0.2043,0.3770,0.5547,0.7815,0.9491,0.9727,0.9248,0.9996,0.9994,0.9667
Coding.Potential_ss_lncfinder,0.0355,0.0271,0.1227,0.0029,0.3131,0.2590,0.3978,0.3327,0.3345,0.7935,1.0000,0.3175,0.6541,0.1856,0.9294,1.0000,0.1656
H_exp,0.0441,0.0609,0.0669,0.0597,0.2195,0.2418,0.2152,0.1684,0.2369,0.5054,0.5764,0.6156,0.5634,0.5310,0.5377,0.5226,0.3517
H_pred,0.0491,0.0691,0.0721,0.0634,0.2636,0.2667,0.2508,0.1863,0.2702,0.8637,0.8937,0.9550,0.9294,0.8591,0.9709,0.8791,0.9928


## 6. Pipeline features by block

Every feature column, grouped into thematic blocks by name prefix. Blocks are derived from prefixes, so features added by new pipelines still appear (anything unmatched lands in **Other**).

In [ ]:
FEATURE_BLOCKS = {
    "Transposable elements — RNA-level":     r"^rna_te_",
    "Transposable elements — DNA-level":     r"^dna_te_",
    "Repeats / low-complexity / pseudogene": r"^(rna|dna)_(lctr|pseudo|global|unknown)",
    "Non-B DNA motifs":                      r"^(apr|dr|gq|ir|mr|str|tri|z)_|nonb",
    "Secondary structure / Z-propensity / rG4": r"^(prop_|avg|rg4_|has_peak|rel_pos|n_|frac_|motif_)",
}

assigned, block_of = set(), {}
for name, pat in FEATURE_BLOCKS.items():
    matched = [c for c in feat_cols if re.search(pat, c)]
    for c in matched:
        block_of.setdefault(c, name)
    assigned.update(matched)
other = [c for c in feat_cols if c not in assigned]
if other:
    for c in other:
        block_of[c] = "Other"

summary = pd.Series(block_of).value_counts().rename("n_features")
print("Features per block:")
print(summary.to_string())

Features per block:
Non-B DNA motifs                            197
Repeats / low-complexity / pseudogene       178
Transposable elements — DNA-level            81
Transposable elements — RNA-level            81
Secondary structure / Z-propensity / rG4     69
Other                                         1


In [ ]:
from IPython.display import display, Markdown

block_order = list(FEATURE_BLOCKS) + (["Other"] if other else [])
for name in block_order:
    cols = [c for c in feat_cols if block_of.get(c) == name]
    if not cols:
        continue
    display(Markdown(f"### {name}  ({len(cols)} features)"))
    display(show(feat[cols]))

### Transposable elements — RNA-level  (81 features)

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
rna_te_min_sw_score,1411.0000,0.0,1411.0000,0.0,0.0,0.0,0.0,0.0,0.0,194.0000,0.0,242.0000,0.0,0.0,0.0,194.0000,203.0000
rna_te_max_sw_score,1411.0000,0.0,1411.0000,0.0,0.0,0.0,0.0,0.0,0.0,333.0000,0.0,242.0000,0.0,0.0,0.0,194.0000,740.0000
rna_te_mean_sw_score,1411.0000,0.0,1411.0000,0.0,0.0,0.0,0.0,0.0,0.0,271.6667,0.0,242.0000,0.0,0.0,0.0,194.0000,408.6667
rna_te_min_divergence,18.1000,0.0,18.1000,0.0,0.0,0.0,0.0,0.0,0.0,20.9000,0.0,20.3000,0.0,0.0,0.0,20.3000,8.4000
rna_te_max_divergence,18.1000,0.0,18.1000,0.0,0.0,0.0,0.0,0.0,0.0,29.8000,0.0,20.3000,0.0,0.0,0.0,20.3000,25.8000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
rna_te_retroposon_count_per_kb,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0000,0.0000
rna_te_sine_count_per_kb,0.1565,0.0,0.1688,0.0,0.0,0.0,0.0,0.0,0.0,1.0817,0.0,0.0000,0.0,0.0,0.0,1.0482,1.4567
rna_te_srprna_count_per_kb,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0000,0.0000
rna_te_young_count_per_kb,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0000,0.0000


### Transposable elements — DNA-level  (81 features)

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
dna_te_min_sw_score,181.0000,312.0000,993.0,276.0000,0.0,0.0,0.0,0.0,0.0,0.0,908.0000,0.0,0.0,0.0,0.0,0.0,283.0000
dna_te_max_sw_score,6043.0000,4531.0000,993.0,2409.0000,0.0,0.0,0.0,0.0,0.0,0.0,908.0000,0.0,0.0,0.0,0.0,0.0,7922.0000
dna_te_mean_sw_score,1187.6552,1399.0000,993.0,1267.2857,0.0,0.0,0.0,0.0,0.0,0.0,908.0000,0.0,0.0,0.0,0.0,0.0,2499.5000
dna_te_min_divergence,2.9000,5.7000,18.6,5.3000,0.0,0.0,0.0,0.0,0.0,0.0,21.7000,0.0,0.0,0.0,0.0,0.0,1.9000
dna_te_max_divergence,42.1000,34.1000,18.6,30.1000,0.0,0.0,0.0,0.0,0.0,0.0,21.7000,0.0,0.0,0.0,0.0,0.0,17.3000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
dna_te_retroposon_count_per_kb,0.0099,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000
dna_te_sine_count_per_kb,0.5947,0.0706,0.0,0.0712,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.1754
dna_te_srprna_count_per_kb,0.0000,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000
dna_te_young_count_per_kb,0.0496,0.0202,0.0,0.0102,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000


### Repeats / low-complexity / pseudogene  (178 features)

...

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
apr_hit_count,20.0000,19.0000,20.0000,18.0000,0.0000,0.000,0.000,0.0000,0.0000,4.0000,1.0000,0.0000,4.0000,0.0000,4.0000,4.000,1.0000
apr_total_length,508.0000,483.0000,508.0000,457.0000,0.0000,0.000,0.000,0.0000,0.0000,102.0000,25.0000,0.0000,102.0000,0.0000,102.0000,102.000,24.0000
apr_max_length,28.0000,28.0000,28.0000,28.0000,0.0000,0.000,0.000,0.0000,0.0000,27.0000,25.0000,0.0000,27.0000,0.0000,27.0000,27.000,24.0000
apr_mean_length,25.4000,25.4211,25.4000,25.3889,0.0000,0.000,0.000,0.0000,0.0000,25.5000,25.0000,0.0000,25.5000,0.0000,25.5000,25.500,24.0000
apr_std_length,1.4654,1.5024,1.4654,1.5392,0.0000,0.000,0.000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,1.000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
all_nonb_gaps_min,-125.0000,-125.0000,-125.0000,-125.0000,-28.0000,-28.000,-28.000,-28.0000,-28.0000,-79.0000,-41.0000,4.0000,-75.0000,4.0000,-75.0000,-75.000,-109.0000
any_nonb_present,1.0000,1.0000,1.0000,1.0000,1.0000,1.000,1.000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.000,1.0000
total_nonb_count,484.0000,471.0000,484.0000,467.0000,21.0000,23.000,15.000,19.0000,8.0000,180.0000,68.0000,14.0000,141.0000,14.0000,131.0000,131.000,265.0000
total_nonb_coverage,12308.0000,12092.0000,12308.0000,12010.0000,348.0000,439.000,259.000,374.0000,159.0000,4013.0000,1430.0000,246.0000,2928.0000,246.0000,2745.0000,2758.000,6558.0000


### Secondary structure / Z-propensity / rG4  (69 features)

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
motif_types_present,9,9,9,9,5,5,5,5,4,9,9,4,9,4,9,9,9
motif_diversity,1.568738,1.582502,1.568738,1.575145,1.353927,1.363188,1.459482,1.294775,1.213008,1.741772,1.76374,1.170997,1.704279,1.170997,1.70645,1.720329,1.755836
avgZ_min,-4.11,-4.11,-4.21,-4.11,-4.12,-4.06,-4.04,-2.59,-2.63,-3.37,-2.52,-3.71,-2.1,-2.47,-3.02,-1.98,-4.45
avgZ_mean,-0.94508,-1.036672,-0.970213,-0.995063,-1.066399,-0.877361,-0.83245,-0.5074,-0.747506,-1.347301,-0.899653,-1.17934,-0.485481,-1.068075,-1.252427,-0.750294,-1.605987
avgZ_median,-0.91,-1.06,-1.01,-1.05,-0.94,-0.77,-0.78,-0.59,-0.91,-1.17,-0.67,-0.84,-0.37,-1.06,-1.17,-0.67,-1.27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
prop_z_lt_minus2_bin10,0.115672,0.0,0.116688,0.441478,0.467532,0.270925,0.310345,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.026316
rg4_peak_count,0,0,0,0,1,3,1,1,1,0,0,0,0,0,0,0,0
rg4_peak_density,0.0,0.0,0.0,0.0,0.000502,0.00093,0.000356,0.000751,0.001144,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rg4_peak_rel_position,-1.0,-1.0,-1.0,-1.0,0.846038,0.782119,0.890825,0.561232,0.508009,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0


### Other  (1 features)

tid,EGF (ENST00000265171 · low_coding),EGF (ENST00000509793 · low_coding),EGF (ENST00000652245 · low_coding),EGF (ENST00000503392 · low_coding),MALAT1 (ENST00000710944 · low_lncRNA),MALAT1 (ENST00000710871 · low_lncRNA),MALAT1 (ENST00000710946 · low_lncRNA),MALAT1 (ENST00000710870 · low_lncRNA),MALAT1 (ENST00000710865 · low_lncRNA),MEG3 (ENST00000650041 · high_lncRNA),MEG3 (ENST00000452514 · high_lncRNA),MEG3 (ENST00000648512 · high_lncRNA),MEG3 (ENST00000648123 · high_lncRNA),MEG3 (ENST00000650556 · high_lncRNA),MEG3 (ENST00000524131 · high_lncRNA),MEG3 (ENST00000522618 · high_lncRNA),SWI5 (ENST00000652598 · high_coding)
unspliced_length,100884,99111,100867,98261,5467,4366,3264,3121,1186,34909,15908,4847,29320,4726,25291,25268,39902


## 7. Compact summary table

One row per transcript: transcript length, gene (unspliced) length, DNA/RNA
repeat hits, non-B DNA (NBD) hits, and rG4 hits — the small table to sit
alongside the SHAP waterfall figures. `feat` (loaded above) has everything
except spliced transcript length, which isn't in `supplementary_features.tsv`
(dropped during the pipeline merge); pulled here from the TE-RNA pipeline's own
`transcript_length` column instead.

In [ ]:
TE_RNA_CSV = BASE / "te_pipeline/results/gencode.v47/features/all_transcripts_te_features.csv"

full_id = groups.set_index("tid")["seq_ID"]  # short tid -> full versioned ENST id

te_rna_len = pd.read_csv(TE_RNA_CSV, usecols=["transcript_id", "transcript_length"])
te_rna_len = te_rna_len.set_index("transcript_id")["transcript_length"]
transcript_length = full_id.reindex(selected.index).map(te_rna_len).rename("transcript_length")

summary = pd.DataFrame({
    "gene": selected["gene"],
    "group": selected["entropy_group"],
    "transcript_length": transcript_length,
    "gene_length": feat["unspliced_length"],
    "dna_rep_hits": feat["dna_te_count"],
    "rna_rep_hits": feat["rna_te_count"],
    "nbd_hits": feat["total_nonb_count"],
    "rg4_hits": feat["rg4_peak_count"],
})

# Only keep TRANSCRIPT_IDS values in the summary DF
summary = summary.loc[list(TRANSCRIPT_IDS.values())]
summary

,gene,group,transcript_length,gene_length,dna_rep_hits,rna_rep_hits,nbd_hits,rg4_hits
tid,,,,,,,,
ENST00000265171,EGF,low_coding,6388,100884,116,1,484.0,0
ENST00000710946,MALAT1,low_lncRNA,2812,3264,0,0,15.0,1
ENST00000652598,SWI5,high_coding,1373,39902,8,3,265.0,0
ENST00000648123,MEG3,high_lncRNA,644,29320,0,0,141.0,0
